# 07 — Quantum Parity-Check Matrices and Decoding

## Read before you begin

- Arthur Pesah, [*Stabilizer Formalism III*](https://arthurpesah.me/blog/2023-03-28-stabilizer-formalism-3/): **Motivation**, **Binary symplectic format**, and **Logical operators**. Extract how a Pauli becomes two binary strings and why commuting is symplectic.

## Learning objectives

- Translate $P$ into $(x\mid z)$.
- Compute commutation and syndromes modulo 2.
- Explain error cosets and read a Tanner graph.

## Prerequisites and how to use this notebook

Know stabilizer generators, CSS codes, and Steane code from notebooks 03–06. Here $n$ counts data qubits and every matrix row is one stabilizer generator.

Run cells in order. Pause at each prediction and make a claim before revealing the output.

**By the end:** you can move between Pauli strings, binary matrices, and decoder graphs; notebook 08 adds noise.

## 1. The binary symplectic representation

Every single-qubit Pauli operator, ignoring its overall phase, is one of four possibilities. Represent each as a pair of bits $(x, z)$:

| Pauli | $x$ | $z$ |
|---|---:|---:|
| $I$ | 0 | 0 |
| $X$ | 1 | 0 |
| $Z$ | 0 | 1 |
| $Y$ | 1 | 1 |

($Y = iXZ$ up to phase, hence $x=z=1$.) An $n$-qubit Pauli string becomes one length-$2n$ binary vector: all the $x$-bits, then all the $z$-bits.

### Code preview — encode Paulis once, then reuse the representation

This cell translates $P\mapsto(x\mid z)$. `Y` has both an $X$ and $Z$ component, so it puts a 1 in both halves. Global phase is ignored because it does not change commutation or syndromes.

In [ ]:
def symplectic_vector(pauli_string):
    x_part = tuple(1 if ch in ("X", "Y") else 0 for ch in pauli_string)
    z_part = tuple(1 if ch in ("Z", "Y") else 0 for ch in pauli_string)
    return x_part + z_part

example_pauli_strings = ["XIYZ", "IXYZ", "XXXI", "ZZZZ"]
for s in example_pauli_strings:
    print(f"{s} -> {symplectic_vector(s)}")

import stim

# stim.PauliString already stores exactly this representation internally --
# to_numpy() returns the same (x-part, z-part) pair as booleans, so this is a
# direct check that the by-hand construction above is the real thing, not a
# lookalike.
for s in example_pauli_strings:
    xs, zs = stim.PauliString(s).to_numpy()
    stim_vector = tuple(int(b) for b in xs) + tuple(int(b) for b in zs)
    print(f"{s}: matches stim.PauliString.to_numpy()?", stim_vector == symplectic_vector(s))

## 2. The symplectic product decides commutation

For two length-$2n$ vectors $v=(x\mid z)$ and $w=(x'\mid z')$, the **symplectic product** is

$$v \odot w = x \cdot z' + z \cdot x' \pmod 2$$

Two Pauli strings commute exactly when their symplectic product is 0. This single linear formula replaces notebooks 05–06's "count disagreements at shared non-identity positions" rule with ordinary linear algebra — let's verify it agrees with `stim.PauliString.commutes()` (the same primitive notebook 06 used) on every pair among the Steane code's 6 generators plus a handful of single-qubit errors.

### Code preview — commutation becomes one bit

The symplectic product returns `0` for commuting and `1` for anticommuting Paulis. `bits_of` is only a display helper for the Steane example.

In [ ]:
def symplectic_product(v, w):
    # check dimension match
    assert len(v) == len(w)
    n = len(v) // 2
    xv, zv, xw, zw = v[:n], v[n:], w[:n], w[n:]
    return (sum(a * b for a, b in zip(xv, zw)) + sum(a * b for a, b in zip(zv, xw))) % 2


def bits_of(j, width):
    # binary form of integer j as a tuple from most significant bit
    return tuple(int(bit) for bit in format(j, f"0{width}b"))


H_columns = [bits_of(j, width=3) for j in range(1, 8)]
H = tuple(tuple(col[row] for col in H_columns) for row in range(3))
steane_generators = (["".join("X" if b else "I" for b in row) for row in H]
                     + ["".join("Z" if b else "I" for b in row) for row in H])

test_strings = steane_generators + ["XIIIIII", "IIIIIIZ", "YIIIIII"]
print("Testing commutation of the following Pauli strings:")
for s in test_strings:
    print(f"  {s}") 

all_agree = True
for p in test_strings:
    for q in test_strings:
        stim_commute = stim.PauliString(p).commutes(stim.PauliString(q))
        symplectic_commute = symplectic_product(symplectic_vector(p), symplectic_vector(q)) == 0
        if stim_commute != symplectic_commute:
            all_agree = False
            print(f"Disagreement: {p} and {q} -> stim: {stim_commute}, symplectic: {symplectic_commute}")

print("stim.PauliString.commutes() and the symplectic product always agree:", all_agree)

### Prediction 1

**Before running the next cell:** notebook 05 built the Steane code's 3 $X$-type generators directly from $H$'s rows (with zero $Z$-part) and its 3 $Z$-type generators from the same rows (with zero $X$-part). If you stack all 6 generators' symplectic vectors into a $6\times 14$ matrix $M$, what should the top-left $3\times 7$ block and bottom-right $3\times 7$ block look like? What should the two *off*-diagonal blocks (top-right, bottom-left) look like?

In [ ]:
M_steane = [list(symplectic_vector(s)) for s in steane_generators]
import numpy as np
M_steane_np = np.array(M_steane)

print("Quantum parity-check matrix M (6 rows, 14 columns = 7 X-columns | 7 Z-columns):")
print(M_steane_np)
print()
print("Top-left block (X-generators' x-part) equals H:", np.array_equal(M_steane_np[:3, :7], np.array(H)))
print("Bottom-right block (Z-generators' z-part) equals H:", np.array_equal(M_steane_np[3:, 7:], np.array(H)))
print("Top-right block (X-generators' z-part) is all zero:", np.all(M_steane_np[:3, 7:] == 0))
print("Bottom-left block (Z-generators' x-part) is all zero:", np.all(M_steane_np[3:, :7] == 0))

This is exactly the **CSS block form** $M = \begin{pmatrix} H_X & 0 \\ 0 & H_Z \end{pmatrix}$, with $H_X = H_Z = H$ for the Steane code specifically. The commutation requirement between every $X$-type and every $Z$-type generator — which notebook 05 checked directly as $H H^T \equiv 0 \pmod 2$ — is exactly the statement that every row of the top block has zero symplectic product with every row of the bottom block. The symplectic formalism doesn't introduce a new fact here; it explains *why* notebook 05's condition was the right one to check.

## 3. Syndromes as a matrix-vector product

An error $e$ (also a Pauli string, converted to its symplectic vector) produces a syndrome by taking its symplectic product against *every row* of $M$ — one bit per stabilizer generator.

### Code preview — a syndrome is a list of commutation tests

For each physical $X$ error, the code asks every stabilizer row whether it anticommutes. Expect the opposite-type checks to contain the useful location pattern.

In [ ]:
def quantum_syndrome(M, error_string, n):
    e = symplectic_vector(error_string)
    return tuple(symplectic_product(row, e) for row in M)


for qubit in range(7):
    error_string = "".join("X" if i == qubit else "I" for i in range(7))
    syndrome = quantum_syndrome(M_steane, error_string, n=7)
    x_part, z_part = syndrome[:3], syndrome[3:]
    print(f"X error on qubit {qubit}: syndrome = {syndrome}  (X-checks: {x_part}, Z-checks: {z_part})")

This matches notebook 05's Section 7 exactly: an $X$ error only ever trips the $Z$-type checks (the first three syndrome bits, from $M$'s top block, are always 0), and the $Z$-check syndrome bits reproduce notebook 04's classical syndrome for that qubit position. The matrix-vector product computes, in one step, what notebook 05 computed by simulating the full 7-qubit state.

## 4. Decoding by error cosets, not exact error location

Two errors $e$ and $e'$ produce the *same* syndrome exactly when $e \oplus e'$ (their symplectic-vector XOR) has zero symplectic product with every row of $M$ — i.e., $e \oplus e'$ is itself undetectable. If, in addition, $e \oplus e'$ lies in $M$'s own row space (some XOR-combination of the stabilizer generators themselves), then $e$ and $e'$ are not just "same syndrome" but **exactly the same physical effect on the code space** — this is notebook 05's degeneracy, restated in linear algebra: a decoder does not need to recover the literal error, only *a* representative of the right coset of the row space.

### Prediction 2

**Before running the next cell:** on the Shor code, a $Z$ error on qubit 0, qubit 1, or qubit 2 (all in block A) were shown in notebook 05 to produce the *identical* corrupted state. In symplectic terms, that means $e_{Z_0} \oplus e_{Z_1}$ should be expressible as an XOR-combination of the Shor code's own stabilizer generators. Which generator(s), specifically?

In [ ]:
shor_generators = ["ZZIIIIIII", "IZZIIIIII", "IIIZZIIII", "IIIIZZIII",
                   "IIIIIIZZI", "IIIIIIIZZ", "XXXXXXIII", "IIIXXXXXX"]

e_z0 = symplectic_vector("ZIIIIIIII")
e_z1 = symplectic_vector("IZIIIIIII")
difference = tuple(a ^ b for a, b in zip(e_z0, e_z1))
z_part = difference[9:]  # X-part of the difference is all zero, since both errors are pure Z
print("e(Z0) XOR e(Z1), Z-part only:", z_part, " <- this should be ZZ on qubits 0,1: 'ZZIIIIIII'")
print("That is exactly the first Shor stabilizer generator:", "".join("Z" if b else "I" for b in z_part) == shor_generators[0])

## 5. Logical operators as kernel elements

A vector $v$ represents a **logical operator** exactly when it is in the *symplectic kernel* of every generator (zero symplectic product with all of them — i.e., it commutes with every stabilizer) but is **not** in $M$'s row space (not itself a product of stabilizers). This is notebook 06's centralizer-minus-stabilizer condition, now phrased as ordinary linear algebra: kernel membership is a set of linear equations over $\mathrm{GF}(2)$, and row-space membership is checking against $2^m$ XOR-combinations of the $m$ generator rows.

### Code preview — find a logical operator in a small example

This brute-force search keeps the mathematical condition visible: commute with every check, but do not lie in the stabilizer row space. Real decoders use GF(2) linear algebra for large codes.

In [ ]:
from itertools import product as iproduct, combinations


def in_row_space(v, M):
    # Brute-force check if v is in the row space of M (over GF(2)),
    # See cell below for a more efficient method using linear algebra.
    m = len(M)
    length = len(v)
    for bits in iproduct([0, 1], repeat=m):
        combo = [0] * length
        for b, row in zip(bits, M):
            if b:
                combo = [c ^ r for c, r in zip(combo, row)]
        if tuple(combo) == tuple(v):
            return True
    return False


def weight_of_symplectic(v, n):
    return sum(1 for i in range(n) if v[i] or v[n + i])


def all_symplectic_weight(w, n):
    """All length-2n symplectic vectors of Pauli weight exactly w (nonzero on w qubits)."""
    results = []
    for positions in combinations(range(n), w):
        for letters in iproduct([(1, 0), (0, 1), (1, 1)], repeat=w):  # X, Z, Y per chosen position
            x = [0] * n
            z = [0] * n
            for pos, (xb, zb) in zip(positions, letters):
                x[pos], z[pos] = xb, zb
            results.append(tuple(x) + tuple(z))
    return results


min_weight_logical = None
for w in range(1, 4):
    found_at_this_weight = []
    for v in all_symplectic_weight(w, n=7):
        commutes_with_all = all(symplectic_product(row, v) == 0 for row in M_steane)
        if commutes_with_all and not in_row_space(v, M_steane):
            found_at_this_weight.append(v)
    if found_at_this_weight:
        min_weight_logical = w
        print(f"First nontrivial logical operator(s) found at weight {w}: {len(found_at_this_weight)} of them")
        break
    else:
        print(f"No nontrivial logical operator at weight {w}")

print("\nSteane code distance, found by pure linear algebra:", min_weight_logical)

In [ ]:
def xor_rows(row_a, row_b):
    """Add two binary rows: 1 + 1 = 0, so use XOR."""
    return tuple(a ^ b for a, b in zip(row_a, row_b))


def first_one(row):
    """Return the column of the first 1, or None for the all-zero row."""
    for column, bit in enumerate(row):
        if bit == 1:
            return column
    return None


def make_basis(rows):
    """
    Turn rows into an independent XOR basis.

    Returns a dictionary:
        pivot column -> one independent row with its first 1 in that column
    """
    basis = {}

    for original_row in rows:
        row = tuple(original_row)
        print(f"Processing row: {row}")

        # Keep cancelling leading 1s using rows already in the basis.
        while True:
            pivot = first_one(row)

            # The row reduced to 000...0.
            # It was already made from earlier basis rows.
            if pivot is None:
                print(f"Row {row} reduced to zero, already in basis.")
                break

            # This leading 1 is new, so this row is independent.
            if pivot not in basis:
                basis[pivot] = row
                print(f"Row {row} is independent, added to basis with pivot {pivot}.")
                break

            # Another basis row has the same leading 1.
            # XOR cancels it.
            row = xor_rows(row, basis[pivot])
            print(f"Row {row} after XOR with basis row at pivot {pivot}.")

    return basis


def in_row_space(v, M):
    """Return True exactly when v can be made by XORing rows of M."""
    basis = make_basis(M)
    print(f"Basis for row space of M: {basis}")
    row = tuple(v)

    # Try to reduce v using the independent basis rows.
    while True:
        print(f"Reducing vector: {row}")
        pivot = first_one(row)

        # v reduced completely to zero:
        # therefore v was a combination of rows of M.
        if pivot is None:
            print(f"Vector {v} reduced to zero, it is in the row space of M.")
            return True

        # No row can cancel this leading 1:
        # therefore v is not in the row space.
        if pivot not in basis:
            print(f"Vector {v} has leading 1 at pivot {pivot}, but no basis row can cancel it. Not in row space.")
            return False

        row = xor_rows(row, basis[pivot])
        print(f"Vector {row} after XOR with basis row {basis[pivot]} at pivot {pivot}.")

M = [
    (1, 0, 1),
    (0, 1, 1),
]

print(in_row_space((1, 1, 0), M))  # True
print('----------------------------------')
print(in_row_space((1, 0, 0), M))  # False

This reproduces notebook 06's distance-3 result for the Steane code — found this time with no state vectors, no $128\times128$ matrices, and no quantum simulation at all, purely by solving a linear-algebra membership question over $\mathrm{GF}(2)$. This is precisely why real decoders (including the one in notebook 08) work with parity-check matrices rather than simulating the full quantum state: the symplectic representation makes every question decoding needs to ask — does this commute? what's the syndrome? is this error degenerate with that one? — a matter of binary linear algebra on vectors of length $2n$, regardless of how large the $2^n$-dimensional state space actually is.

## 6. Reading a parity-check matrix as a Tanner graph

A parity-check matrix is a bipartite graph in disguise: one node per qubit, one node per check, and an edge whenever a check's row has a 1 in that qubit's column. This is exactly how modern decoders (including the one notebook 10 builds around Stim's detector data) visualize which physical qubits feed into which measurement.

### Code preview — expose the graph inside the matrix

Each printed line names the data-qubit neighbors of one check node. This sparse bipartite structure is what a Tanner-graph decoder uses.

In [ ]:
def print_tanner_graph(H, check_prefix="check"):
    for i, row in enumerate(H):
        touched_qubits = [q for q, bit in enumerate(row) if bit]
        print(f"{check_prefix}{i} -- qubits {touched_qubits}")


print("Hamming/Steane H as a Tanner graph:")
print_tanner_graph(H)

Each check (row) connects to exactly 4 of the 7 qubits, and — as Section 3 showed — the same bipartite structure serves as *both* the $X$-check graph and the $Z$-check graph for the Steane code, since both reuse $H$.

## Can you now explain? — Exercises

1. Why are two binary halves needed for a Pauli?
2. Why does a zero syndrome not prove no error occurred?
3. Distinguish a stabilizer, a logical operator, and two errors in one syndrome coset.
4. What does one edge in a Tanner graph mean?

Then try the implementation exercises below.

In [ ]:
def parity_check_matrix(generator_strings, n):
    pass  # TODO: implement, then remove this stub body


def syndrome_coset_size(M, syndrome, n):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# M_shor = parity_check_matrix(shor_generators, n=9)
# for row in M_shor:
#     print(row)
# print(syndrome_coset_size(M_steane, (0, 0, 0, 0, 0, 0), 7))